<a href="https://colab.research.google.com/github/carolinewan/3m-data-assignment-1.1/blob/main/mnist_starter_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST CNN Improvement Workshop - Starter Notebook
## From Lesson 3.7 Neural Network & Deep Learning

This notebook provides the foundation code for implementing CNN improvements on MNIST digit classification.

**Baseline Goal:** Start with MLP achieving ~97% accuracy  
**Workshop Goal:** Achieve 99%+ accuracy using CNN techniques

Please use Conda DL environment with PyTorch installed.

## 1. Import Required Libraries

In [1]:
# Import libraries for deep learning and data handling
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
import numpy as np

# Check for CUDA availability and use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'CUDA device name: {torch.cuda.get_device_name()}')
    print(f'CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Using device: cpu


## 2. Load MNIST Dataset

In [2]:
# Define data preprocessing transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Load training data (60,000 images)
trainset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

# Load test data (10,000 images)
testset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

print(f'Training samples: {len(trainset)}')
print(f'Test samples: {len(testset)}')

100%|██████████| 9.91M/9.91M [00:00<00:00, 59.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.78MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.7MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.41MB/s]

Training samples: 60000
Test samples: 10000


## 3. Helper Functions for Training and Evaluation

In [6]:
def train_model(model, trainloader, criterion, optimizer, scheduler=None, epochs=5):
    """Train a model and return training history"""
    model.train()
    train_losses = []

    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(trainloader)
        train_losses.append(epoch_loss)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}')

        if scheduler:
            scheduler.step()

    return train_losses

In [ ]:
def evaluate_model(model, testloader):
    """Evaluate model and return accuracy"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [7]:
def count_parameters(model):
    """Count total trainable parameters in model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def validate_model_architecture(model):
    """Validate student model meets requirements"""
    param_count = count_parameters(model)
    print(f"Model has {param_count:,} trainable parameters")

    # Test forward pass
    test_input = torch.randn(1, 1, 28, 28).to(device)
    try:
        output = model(test_input)
        assert output.shape == (1, 10), f"Expected output shape (1, 10), got {output.shape}"
        print("✓ Model architecture validation passed")
        return True
    except Exception as e:
        print(f"✗ Model architecture validation failed: {e}")
        return False

## 4. Baseline Model - From Lesson 3.7 Neural Network & Deep Learning

This is our starting point: a Multi-Layer Perceptron (MLP) that achieves approximately 97% accuracy.

In [8]:
class BaselineMLP(nn.Module):
    def __init__(self):
        super(BaselineMLP, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.shape[0], -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

## 5. Train and Evaluate Baseline Model

In [ ]:
print("Training Baseline MLP (from Lesson 3.7 Neural Network & Deep Learning)...")
baseline_model = BaselineMLP().to(device)
baseline_criterion = nn.NLLLoss()
baseline_optimizer = optim.Adam(baseline_model.parameters(), lr=0.003)

# Validate architecture
validate_model_architecture(baseline_model)

# Train the model
baseline_losses = train_model(baseline_model, trainloader, baseline_criterion, baseline_optimizer)

# Evaluate the model
baseline_accuracy = evaluate_model(baseline_model, testloader)
print(f'\nBaseline MLP Accuracy: {baseline_accuracy:.2f}%')

## 6. Improvement Exploration

Now that you have a working baseline achieving ~97% accuracy, explore ways to improve performance.

**Possible improvement paths:**

Architecture Changes, Regularization Approaches, Training Enhancements, Data Augmentation, Others

## 7. Your Implementation Area

Use the cells below to implement your chosen improvement techniques:

In [ ]:
# Implement your improved model here
class ImprovedModel(nn.Module):
    def __init__(self):
        super(ImprovedModel, self).__init__()
        # TODO: Design your architecture, eg. Conv2d...
        pass

    def forward(self, x):
        # TODO: Implement forward pass
        # Consider: proper data flow, activation functions, output format
        pass

In [ ]:
# Test your implementation here
# model = ImprovedModel().to(device)
# validate_model_architecture(model)
#
# criterion = nn.NLLLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.003)
#
# print("Training your improved model...")
# losses = train_model(model, trainloader, criterion, optimizer)
# accuracy = evaluate_model(model, testloader)
# print(f'Your Model Accuracy: {accuracy:.2f}%')

## 8. Results Comparison

In [ ]:
# Compare your results with the baseline
print("\n" + "="*50)
print("ACCURACY COMPARISON")
print("="*50)
print(f"Baseline MLP:        {baseline_accuracy:.2f}%")
# print(f"Your Model:          {accuracy:.2f}%")
# print(f"Improvement:         +{accuracy - baseline_accuracy:.2f}%")
print("="*50)

## 9. Next Steps

**Experiment and iterate:**
- Try different combinations of techniques
- Analyze what works and what doesn't
- Compare training time vs. accuracy trade-offs
- Document your findings and insights

**Remember:** The goal is to understand how different techniques contribute to improved performance!